In [1]:
# 1) Load Packages, Paths, and Global Config
import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Workspace paths
BASE_DIR = Path('.')
OUTPUT_DIR = BASE_DIR

# Robust autodiscovery of train/test CSVs (handles spaces/parentheses)
def _normalize_name(s: str) -> str:
    # lower, remove spaces and parentheses
    return re.sub(r"[\s()]+", "", s.lower())





In [2]:
# 2) Load Train/Test Data
train = pd.read_csv("/kaggle/input/global-temperature-forecasting-1961-2030/train .csv")
test = pd.read_csv("/kaggle/input/global-temperature-forecasting-1961-2030/test ().csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain head:")
display(train.head())
print("\nTrain info:")
print(train.info())
print("\nMissing values in train:")
print(train.isna().sum())

print("\nTest head:")
display(test.head())
print("\nTest info:")
print(test.info())
print("\nMissing values in test:")
print(test.isna().sum())

# Ensure required columns exist or raise clear error
required_train_cols = {'Area','Months','Year','Value'}
missing_train = required_train_cols - set(train.columns)
if missing_train:
    raise ValueError(f"Train CSV is missing required columns: {missing_train}")

required_test_cols = {'ID','Area','Year'}
missing_test = required_test_cols - set(test.columns)
if missing_test:
    raise ValueError(f"Test CSV is missing required columns: {missing_test}")


Train shape: (13607, 14)
Test shape: (1482, 3)

Train head:


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Months Code,Months,Year Code,Year,Unit,Value,Flag,Flag Description
0,ET,Temperature change on land,4,Afghanistan,7271,Temperature change,7020,Meteorological year,1961,1961,°c,-0.126,E,Estimated value
1,ET,Temperature change on land,4,Afghanistan,7271,Temperature change,7020,Meteorological year,1962,1962,°c,-0.173,E,Estimated value
2,ET,Temperature change on land,4,Afghanistan,7271,Temperature change,7020,Meteorological year,1963,1963,°c,0.844,E,Estimated value
3,ET,Temperature change on land,4,Afghanistan,7271,Temperature change,7020,Meteorological year,1964,1964,°c,-0.751,E,Estimated value
4,ET,Temperature change on land,4,Afghanistan,7271,Temperature change,7020,Meteorological year,1965,1965,°c,-0.220,E,Estimated value



Train info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13607 entries, 0 to 13606
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       13607 non-null  object 
 1   Domain            13607 non-null  object 
 2   Area Code (M49)   13607 non-null  int64  
 3   Area              13607 non-null  object 
 4   Element Code      13607 non-null  int64  
 5   Element           13607 non-null  object 
 6   Months Code       13607 non-null  int64  
 7   Months            13607 non-null  object 
 8   Year Code         13607 non-null  int64  
 9   Year              13607 non-null  int64  
 10  Unit              13607 non-null  object 
 11  Value             13607 non-null  float64
 12  Flag              13607 non-null  object 
 13  Flag Description  13607 non-null  object 
dtypes: float64(1), int64(5), object(8)
memory usage: 1.5+ MB
None

Missing values in train:
Domain Code         0
Domain     

,ID,Area,Year
0,1,Afghanistan,2025
1,2,Afghanistan,2026
2,3,Afghanistan,2027
3,4,Afghanistan,2028
4,5,Afghanistan,2029



Test info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1482 entries, 0 to 1481
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      1482 non-null   int64 
 1   Area    1482 non-null   object
 2   Year    1482 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 34.9+ KB
None

Missing values in test:
ID      0
Area    0
Year    0
dtype: int64


In [3]:
# 3) Clean and Select Annual and Seasonal Records
train_clean = train[['Area','Months','Year','Value']].copy()
train_clean['Value'] = pd.to_numeric(train_clean['Value'], errors='coerce')
train_clean = train_clean.dropna(subset=['Value'])
train_clean['Year'] = train_clean['Year'].astype(int)

annual = train_clean[train_clean['Months'] == 'Meteorological year'].copy()
seasonal = train_clean[train_clean['Months'].isin(['Winter','Spring','Summer','Autumn'])].copy()

print('Annual records:', annual.shape)
print('Seasonal records:', seasonal.shape)


Annual records: (13607, 4)
Seasonal records: (0, 4)


In [4]:
# 4) Encode Categorical Features and Build Country Metadata
# Label-encode Area for potential fast joins; keep original string too
area_le = LabelEncoder()
annual['Area_code'] = area_le.fit_transform(annual['Area'])
seasonal['Area_code'] = area_le.transform(seasonal['Area'])

# One-hot encode Months for seasonal modeling later (if needed)
seasonal_ohe = pd.get_dummies(seasonal[['Area','Year','Months']], columns=['Months'])
print('Unique areas:', annual['Area'].nunique())


Unique areas: 247


In [5]:
# 5) Per-Country Time Indexing and Sorting
# Keep only years >= 1961 and sort
annual = annual.loc[annual['Year'] >= 1961].copy()
annual = annual.sort_values(['Area','Year'])

# Drop duplicates if any
annual = annual.drop_duplicates(subset=['Area','Year'], keep='last')

print('Annual year range:', annual['Year'].min(), '->', annual['Year'].max())


Annual year range: 1961 -> 2023


In [6]:
# 6) Feature Engineering: Time, Lags, and Rolling Windows
from typing import List, Dict
from collections import defaultdict


def add_annual_features(df: pd.DataFrame, lags: List[int] = [1,2,3], roll_windows: List[int] = [3,5]) -> pd.DataFrame:
    out = df.copy()
    # Lags
    for L in lags:
        out[f'lag_{L}'] = out.groupby('Area')['Value'].shift(L)
    # Rolling means on Value
    for w in roll_windows:
        out[f'roll_mean_{w}'] = out.groupby('Area')['Value'].rolling(w).mean().reset_index(level=0, drop=True)
    # Rolling slope (last w years) via simple linear regression on (t, Value)
    for w in roll_windows:
        name = f'roll_slope_{w}'
        vals = []
        for area, g in out.groupby('Area', sort=False):
            y = g['Value'].values
            x = np.arange(len(g))
            res = np.full(len(g), np.nan)
            if len(g) >= w:
                for i in range(w-1, len(g)):
                    xi = x[i-w+1:i+1].reshape(-1,1)
                    yi = y[i-w+1:i+1]
                    try:
                        beta = np.polyfit(xi.flatten(), yi, deg=1)
                        res[i] = beta[0]
                    except Exception:
                        res[i] = np.nan
            vals.extend(res.tolist())
        out[name] = vals
    # Deltas
    out['delta'] = out['Value'] - out['lag_1']
    # Year features
    out['Year_centered'] = out['Year'] - 2000
    out['Year_sq'] = out['Year_centered'] ** 2
    return out

annual_fe = add_annual_features(annual)

# Impute initial lag NaNs to allow inclusion (forward/back fill then mean)
for lag_col in ['lag_1','lag_2','lag_3']:
    annual_fe[lag_col] = annual_fe.groupby('Area')[lag_col].transform(lambda s: s.fillna(method='bfill').fillna(method='ffill').fillna(s.mean()))

print('Engineered annual columns:', [c for c in annual_fe.columns if c not in ['Area','Months','Year','Value','Area_code']][:10], '...')


Engineered annual columns: ['lag_1', 'lag_2', 'lag_3', 'roll_mean_3', 'roll_mean_5', 'roll_slope_3', 'roll_slope_5', 'delta', 'Year_centered', 'Year_sq'] ...


In [7]:
# 7) Feature Engineering: Seasonal Signals and Interactions
# Pivot seasonal data to wide format with 4 columns per Area-Year
if not seasonal.empty:
    seasonal_wide = seasonal.pivot_table(index=['Area','Year'], columns='Months', values='Value', aggfunc='mean').reset_index()
    seasonal_wide.columns.name = None
    # Ensure all seasons exist as columns
    for s in ['Winter','Spring','Summer','Autumn']:
        if s not in seasonal_wide.columns:
            seasonal_wide[s] = np.nan
else:
    seasonal_wide = pd.DataFrame(columns=['Area','Year','Winter','Spring','Summer','Autumn'])

# Join seasonal features into annual_fe
annual_all = annual_fe.merge(seasonal_wide, on=['Area','Year'], how='left')

# Interactions
for s in ['Winter','Spring','Summer','Autumn']:
    annual_all[f'Yearx_{s}'] = annual_all['Year_centered'] * annual_all[s]

print('Joined seasonal columns added. Current shape:', annual_all.shape)


Joined seasonal columns added. Current shape: (13607, 23)


In [8]:
# 8) Train/Validation Split with TimeSeriesSplit (<=2014 vs 2015–2024)
VAL_START, VAL_END = 2015, 2024

def split_train_val(df_area: pd.DataFrame):
    train_mask = df_area['Year'] <= 2014
    val_mask = (df_area['Year'] >= VAL_START) & (df_area['Year'] <= VAL_END)
    return df_area.loc[train_mask], df_area.loc[val_mask]

# Feature set to use (drop target and non-features)
TARGET = 'Value'
NON_FEATURES = {'Area','Months','Year','Value','Area_code'}

# We will dynamically drop columns still containing NaNs after creation (per-area)


In [9]:
# 9) Baselines: Naive Last, Mean, Linear Trend

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

baseline_scores = []

for area, g in annual_all.groupby('Area'):
    g = g.sort_values('Year')
    train_g, val_g = split_train_val(g)
    if len(val_g) == 0 or len(train_g) < 5:
        continue
    # Naive last
    last_val = train_g.iloc[-1]['Value']
    y_pred_last = np.full(len(val_g), last_val)
    # Mean
    mean_val = train_g['Value'].mean()
    y_pred_mean = np.full(len(val_g), mean_val)
    # Linear trend on Year only
    lin = LinearRegression()
    lin.fit(train_g[['Year']], train_g['Value'])
    y_pred_lin = lin.predict(val_g[['Year']])

    s_last = rmse(val_g['Value'].values, y_pred_last)
    s_mean = rmse(val_g['Value'].values, y_pred_mean)
    s_lin = rmse(val_g['Value'].values, y_pred_lin)

    baseline_scores.append({
        'Area': area,
        'RMSE_last': s_last,
        'RMSE_mean': s_mean,
        'RMSE_linear_year': s_lin
    })

baseline_df = pd.DataFrame(baseline_scores)
print('Baseline sample:')
display(baseline_df.head())
print('Baseline RMSE (median):', baseline_df[['RMSE_last','RMSE_mean','RMSE_linear_year']].median())


Baseline sample:


,Area,RMSE_last,RMSE_mean,RMSE_linear_year
0,Afghanistan,1.053634,1.142448,0.477368
1,Albania,0.480412,1.375478,0.590777
2,Algeria,0.437823,1.141355,0.385008
3,American Samoa,0.227572,0.698415,0.171583
4,Andorra,0.569253,1.616906,0.718087


Baseline RMSE (median): RMSE_last           0.476945
RMSE_mean           0.954052
RMSE_linear_year    0.415266
dtype: float64


In [10]:
# 10) Model Zoo per Country: LR, Ridge(poly), Lasso, RF, SVR, GBDT

def build_models():
    models = {
        'LR': Pipeline([
            ('select', 'passthrough'),
            ('reg', LinearRegression())
        ]),
        'RidgePoly2': Pipeline([
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('scaler', StandardScaler(with_mean=False)),
            ('reg', Ridge(alpha=1.0, random_state=RANDOM_STATE))
        ]),
        'LassoPoly2': Pipeline([
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('scaler', StandardScaler(with_mean=False)),
            ('reg', Lasso(alpha=0.01, random_state=RANDOM_STATE, max_iter=15000))
        ]),
        'RF': RandomForestRegressor(n_estimators=250, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1),
        'SVRrbf': Pipeline([
            ('scaler', StandardScaler()),
            ('reg', SVR(C=2.0, epsilon=0.1, kernel='rbf'))
        ]),
        'GBDT': GradientBoostingRegressor(n_estimators=350, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE)
    }
    return models

# Helper to get feature columns excluding target/non-features
ALL_CANDIDATE_FEATURES = [c for c in annual_all.columns if c not in NON_FEATURES]

area_model_scores = []
area_model_objects = {}

for area, g in annual_all.groupby('Area'):
    g = g.sort_values('Year')
    train_g, val_g = split_train_val(g)
    if len(val_g) == 0 or len(train_g) < 8:
        continue
    # Row-wise NaN handling rather than dropping whole feature columns
    feat_cols_raw = [c for c in ALL_CANDIDATE_FEATURES if c in train_g.columns]
    train_rows = train_g.dropna(subset=feat_cols_raw)
    val_rows = val_g.dropna(subset=feat_cols_raw)
    if len(train_rows) < 0.7 * len(train_g):
        feat_cols = ['Year_centered','Year_sq'] if 'Year_sq' in train_g.columns else ['Year_centered']
        X_tr, y_tr = train_g[feat_cols], train_g[TARGET]
        X_va, y_va = val_g[feat_cols], val_g[TARGET]
    else:
        feat_cols = feat_cols_raw
        X_tr, y_tr = train_rows[feat_cols], train_rows[TARGET]
        X_va, y_va = val_rows[feat_cols], val_rows[TARGET]

    models = build_models()
    for name, model in models.items():
        try:
            model.fit(X_tr, y_tr)
            preds = model.predict(X_va)
            score = rmse(y_va.values, preds)
            area_model_scores.append({'Area': area, 'model': name, 'rmse': score, 'n_features': len(feat_cols)})
        except Exception as e:
            area_model_scores.append({'Area': area, 'model': name, 'rmse': np.inf, 'n_features': len(feat_cols), 'err': str(e)})

model_scores_df = pd.DataFrame(area_model_scores)
print('Model score sample:')
display(model_scores_df.head())


Model score sample:


,Area,model,rmse,n_features
0,Afghanistan,LR,0.432705,2
1,Afghanistan,RidgePoly2,0.573510,2
2,Afghanistan,LassoPoly2,0.451113,2
3,Afghanistan,RF,0.921135,2
4,Afghanistan,SVRrbf,1.056428,2


In [11]:
# 11) Hyperparameter Tuning (light GridSearch) with TimeSeries CV
param_grids = {
    'RidgePoly2': {'reg__alpha': [0.1, 1.0, 3.0]},  # degree fixed at 2
    'LassoPoly2': {'reg__alpha': [0.005, 0.01, 0.05]},
    'SVRrbf': {'reg__C': [1.0, 2.0], 'reg__epsilon': [0.05, 0.1]},
    'RF': {'n_estimators': [250,400], 'max_depth': [5,6]},
    'GBDT': {'n_estimators': [300,400], 'learning_rate': [0.05], 'max_depth': [2,3]}
}

best_by_area = []

for area, g in annual_all.groupby('Area'):
    g = g.sort_values('Year')
    train_g, val_g = split_train_val(g)
    if len(val_g) == 0 or len(train_g) < 12:
        continue
    feat_cols_raw = [c for c in ALL_CANDIDATE_FEATURES if c in train_g.columns]
    train_rows = train_g.dropna(subset=feat_cols_raw)
    val_rows = val_g.dropna(subset=feat_cols_raw)
    if len(train_rows) < 0.7 * len(train_g):
        feat_cols = ['Year_centered','Year_sq'] if 'Year_sq' in train_g.columns else ['Year_centered']
        X_tr, y_tr = train_g[feat_cols], train_g[TARGET]
        X_va, y_va = val_g[feat_cols], val_g[TARGET]
    else:
        feat_cols = feat_cols_raw
        X_tr, y_tr = train_rows[feat_cols], train_rows[TARGET]
        X_va, y_va = val_rows[feat_cols], val_rows[TARGET]

    models = build_models()
    best_name, best_model, best_rmse = None, None, np.inf

    for name, model in models.items():
        try:
            if name in param_grids:
                tscv = TimeSeriesSplit(n_splits=min(3, max(2, len(X_tr)//10)))
                grid = GridSearchCV(model, param_grids[name], scoring='neg_root_mean_squared_error', cv=tscv, n_jobs=-1)
                grid.fit(X_tr, y_tr)
                cand = grid.best_estimator_
            else:
                cand = model
                cand.fit(X_tr, y_tr)
            preds = cand.predict(X_va)
            s = rmse(y_va.values, preds)
            if s < best_rmse:
                best_name, best_model, best_rmse = name, cand, s
        except Exception as e:
            continue

    best_by_area.append({'Area': area, 'best_model': best_name, 'val_RMSE': best_rmse, 'features_used': feat_cols})

best_df = pd.DataFrame(best_by_area).sort_values('val_RMSE')
print('Best model per Area (sample):')
display(best_df.head(20))


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:631: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.432e-03, tolerance: 5.222e-04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:631: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.734e-03, tolerance: 4.552e-04
  model = cd_fast.enet_coordinate_descent(


Best model per Area (sample):


,Area,best_model,val_RMSE,features_used
40,China,LR,0.104471,"[Year_centered, Year_sq]"
44,"China, mainland",LR,0.104589,"[Year_centered, Year_sq]"
116,Liberia,LR,0.117085,"[Year_centered, Year_sq]"
185,Solomon Islands,LR,0.121089,"[Year_centered, Year_sq]"
218,Wake Island,LassoPoly2,0.136626,"[Year_centered, Year_sq]"
33,Cameroon,LassoPoly2,0.152853,"[Year_centered, Year_sq]"
124,Maldives,LR,0.154518,"[Year_centered, Year_sq]"
103,Italy,LR,0.168290,"[Year_centered, Year_sq]"
128,Martinique,LR,0.171508,"[Year_centered, Year_sq]"
3,American Samoa,LassoPoly2,0.172416,"[Year_centered, Year_sq]"


In [12]:
# 12) Metric Computation and Model Selection per Country (RMSE)
# Already computed in best_df; summarize
print('Validation RMSE summary (2015–2024):')
print(best_df['val_RMSE'].describe())

# Build a lookup for best model choice and feature columns per area
best_lookup = {row.Area: (row.best_model, row.features_used) for row in best_df.itertuples(index=False)}


Validation RMSE summary (2015–2024):
count    223.000000
mean       0.379018
std        0.183030
min        0.104471
25%        0.256331
50%        0.341770
75%        0.460764
max        1.317609
Name: val_RMSE, dtype: float64


In [13]:
# 13) Retrain Best Models on Full History and Forecast 2025–2030
PRED_YEARS = [2025,2026,2027,2028,2029,2030]

# Utility to rebuild features for a given area frame with appended future rows
FEATURE_BASE = [c for c in annual_all.columns if c not in NON_FEATURES]

from copy import deepcopy

def forecast_area(area: str) -> pd.DataFrame:
    g = annual_all[annual_all['Area'] == area].sort_values('Year').copy()
    # Determine features to use and best model name
    if area in best_lookup:
        best_name, feat_cols = best_lookup[area]
    else:
        best_name, feat_cols = 'LR', ['Year_centered']

    models = build_models()
    model = deepcopy(models.get(best_name, models['LR']))

    # Fit on all history (<=2024) with clean feature set
    g_hist = g[g['Year'] <= 2024].copy()
    # Drop columns with NaNs in history subset
    feat_cols = [c for c in feat_cols if c in g_hist.columns and not g_hist[c].isna().any()]
    if not feat_cols:
        feat_cols = ['Year_centered']

    X_hist, y_hist = g_hist[feat_cols], g_hist[TARGET]
    model.fit(X_hist, y_hist)

    # Iteratively roll forward 2025–2030 using predicted values to fill lags/rolling
    g_future = []
    last_known = g.copy()
    for y in PRED_YEARS:
        row = {
            'Area': area,
            'Months': 'Meteorological year',
            'Year': y,
            # Value unknown; set NaN initially
            'Value': np.nan,
            'Area_code': last_known['Area_code'].iloc[0]
        }
        # Append and rebuild engineered features only for this area
        tmp = pd.concat([last_known, pd.DataFrame([row])], ignore_index=True)
        tmp = add_annual_features(tmp)
        # Attach seasonal wide values for year y if available (else NaN)
        sw = seasonal_wide[seasonal_wide['Area'] == area]
        tmp = tmp.merge(sw, on=['Area','Year'], how='left')
        for s in ['Winter','Spring','Summer','Autumn']:
            tmp[f'Yearx_{s}'] = tmp['Year_centered'] * tmp[s]
        # Select the last row (current future year)
        cur = tmp[tmp['Year'] == y].iloc[-1:].copy()
        # Choose usable features (no NaNs); fallback minimal set
        feat_y_cols = [c for c in feat_cols if c in cur.columns and not cur[c].isna().any()]
        if not feat_y_cols:
            feat_y_cols = ['Year_centered']
        y_pred = model.predict(cur[feat_y_cols])[0]
        row['Value'] = float(y_pred)
        g_future.append(row)
        # Update last_known with the predicted value for next iteration
        last_known = pd.concat([last_known, pd.DataFrame([row])], ignore_index=True)

    return pd.DataFrame(g_future)

# Run forecasting per area
all_forecasts = []
for area in sorted(test['Area'].unique()):
    try:
        f = forecast_area(area)
        all_forecasts.append(f)
    except Exception as e:
        # Fallback: simple linear regression on Year only
        g = annual_all[annual_all['Area'] == area].sort_values('Year')
        lin = LinearRegression().fit(g[g['Year']<=2024][['Year']], g[g['Year']<=2024]['Value'])
        f = pd.DataFrame({
            'Area': area,
            'Months': 'Meteorological year',
            'Year': PRED_YEARS,
            'Value': lin.predict(pd.DataFrame({'Year': PRED_YEARS}))
        })
        all_forecasts.append(f)

forecast_df = pd.concat(all_forecasts, ignore_index=True)
print('Forecast sample:')
display(forecast_df.head())



Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGELSD.

Intel oneMKL ERROR: Parameter 6 was incorrect on entry to DGE

,Area,Months,Year,Value
0,Afghanistan,Meteorological year,2025,1.547649
1,Afghanistan,Meteorological year,2026,1.577178
2,Afghanistan,Meteorological year,2027,1.606708
3,Afghanistan,Meteorological year,2028,1.636237
4,Afghanistan,Meteorological year,2029,1.665767


In [14]:
# 14) Ensembling per Country (optional)
# Weighted by inverse RMSE among top-2 models (if available in best_df)
# Note: For simplicity, we already use the best model. Implementing full ensembling
# would require storing 2nd-best models and refitting; this cell documents the approach.
print('Ensembling step documented; best-model forecasts used for submission.')


Ensembling step documented; best-model forecasts used for submission.


In [15]:
# 15) Generate and Validate Submission CSV
# Map predictions to test rows
pred_map = forecast_df.set_index(['Area','Year'])['Value'].to_dict()

submission = test[['ID','Area','Year']].copy()
missing_keys = []
vals = []
for r in submission.itertuples(index=False):
    key = (r.Area, int(r.Year))
    v = pred_map.get(key)
    if v is None or pd.isna(v):
        missing_keys.append(key)
        # Fallback: linear trend on Year
        g = annual_all[annual_all['Area'] == r.Area].sort_values('Year')
        lin = LinearRegression().fit(g[g['Year']<=2024][['Year']], g[g['Year']<=2024]['Value'])
        v = float(lin.predict([[int(r.Year)]])[0])
    vals.append(v)

submission['Predicted_Anomaly'] = np.round(vals, 3)

# Validate
assert submission['ID'].isna().sum() == 0
assert submission['Predicted_Anomaly'].isna().sum() == 0
print('Submission rows:', len(submission))
print(submission.head())

out_path = OUTPUT_DIR / 'submission_best_models_optimized.csv'
submission[['ID','Predicted_Anomaly']].to_csv(out_path, index=False)
print('Saved submission to:', out_path)


Submission rows: 1482
   ID         Area  Year  Predicted_Anomaly
0   1  Afghanistan  2025              1.548
1   2  Afghanistan  2026              1.577
2   3  Afghanistan  2027              1.607
3   4  Afghanistan  2028              1.636
4   5  Afghanistan  2029              1.666
Saved submission to: submission_best_models_optimized.csv


In [17]:
# (New) Feature Usage Analysis
# Summarize frequency of features among chosen best models
if 'best_df' in globals():
    feature_usage = {}
    for row in best_df.itertuples(index=False):
        for f in row.features_used:
            feature_usage[f] = feature_usage.get(f, 0) + 1
    usage_df = (pd.Series(feature_usage, name='count').sort_values(ascending=False).to_frame())
    print('Top 25 features by usage across best models:')
    display(usage_df.head(25))
else:
    print('best_df not defined yet.')


Top 25 features by usage across best models:


,count
Year_centered,223
Year_sq,223
